In [1]:
from utils import hash_func
import numpy as np
import csv

def count_sketch(filepath, limit, r, b, eps):
    '''
    Description:
        Runs count sketch algorithm on IP addresses
    Inputs:
        filepath: file to run algorithm on
        limit: number of addresses to process
        r: number of algorithm repetitions
        b: number of buckets (hyperparam)
        eps: desired accuracy (hyperparam)
    Outputs:
        returns list of heavy hitters for filepath
    '''
    
    heavy_hitters = []
    buckets_list = np.zeros((r, b))
    approx_norms = np.zeros(r)
    unique_ips = set()
    
    with open(filepath) as csvfile:
        stream = csv.reader(csvfile, delimiter=",")
        stream.__next__() # get rid of the header
        
        for row in stream:
            if (stream.line_num == limit):
                break
            try:
                s_i = int(row[0])
            except:
                # row[0] is not a valid integer
                continue
            unique_ips.add(s_i)
            # process the stream
            for i in range(r):
                curr_buckets = buckets_list[i]
                v_i = 2*hash_func(i+100, s_i, 2) - 1
                bucket = hash_func(i, s_i, b)
                curr_buckets[bucket] += v_i
                approx_norms[i] += v_i
            # determine the threshold
        thresh = eps * np.sqrt(np.median(approx_norms**2))
        # determine the heavy hitters
        for ip in unique_ips:
            f_ip = []
            for i in range(r):
                sign = 2*hash_func(i+100, ip, 2) - 1
                bucket = hash_func(i, ip, b)
                f_ip.append(buckets_list[i][bucket]*sign)
            est_norm = np.median(f_ip)
            if (est_norm > thresh):
                heavy_hitters.append(ip)
        return heavy_hitters

## AOL Dataset

In [4]:
hh_cs = count_sketch(filepath="aol-processed.csv",
                     limit=int(1e4),
                     r=5,
                     b=300,
                     eps=0.1)

In [5]:
hh_cs = sorted(hh_cs)
len(hh_cs)

56